MIT License - Non-LLM

Copyright(c) [2026] [Daksh Gupta. Code.Dkg@gmail.com ]

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The code MUST not be used for any LLM training either standalong or in combination with any other data. The code can be used for any other purpose including but not limited to research, development, testing, and deployment of non-LLM applications.

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.




<br/><br/><br/><br/>

### Vector Database for Metadata

___In this notebook we have a employee table with columns name, designation and salary.___

___Instead of vectorizing the entire employee table, we will vectorize the metadata of the columns and store it in a vector database so that we can perform a similarity search on the metadata to find the most relevant column for a given question.___

<br/><br/>

In [1]:
import requests
import pandas as pd
from pymilvus import MilvusClient, DataType

<br/><br/>
___Here is how the simple metadata will look like. I've added a simple and commonly used description for each of the columns.___
<br/><br/>

In [2]:
columns_metadata = [
    {
        "id": 1,
        "column_name": "salary",
        "description": "salary: employee compensation, annual pay, wages, earnings, money, income amounts"
    },
    {
        "id": 2,
        "column_name": "name",
        "description": "name: employee person name, individual identity, staff names, who people are"
    },
    {
        "id": 3,
        "column_name": "designation",
        "description": "designation: job title, role, position, work title, occupation, department role"
    }
]

<br/><br/>

___We will create a milvus lite vector database for storing the vectorization of the metadata. We will use the ollama local model with [[ nomic-embed-text ]] to get the vector embeddings for the metadata description.___

<br/><br/>

In [3]:
# Create a milvus lite vector database for storing the vectorization of the metadata
client = MilvusClient(uri='meta_employee_data.db')

<br/><br/>
___We will create a schema for the vector database with the following fields:___
- _column_name_
- _description_
- _vector (the vector embedding for the description)_

___We will not use the column_name for vectorization, as it will be part of the description.___
<br/>

___The dimension of the vector embedding will be 768, as we are using the nomic-embed-text model for vectorization.___

<br/><br/>

In [4]:
schema = client.create_schema(auto_id=False)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="column_name", datatype=DataType.VARCHAR, max_length=50)
schema.add_field(field_name="description", datatype=DataType.VARCHAR, max_length=300)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=768)

{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'column_name', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 50}}, {'name': 'description', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 300}}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'enable_dynamic_field': False, 'enable_namespace': False}

<br/><br/>
___Now we'll create an index for the vector field in the schema, we will use FLAT index type with COSINE metric type for similarity search. You can use any other similarity search index type and metric type as per your requirement and the capabilities of the vector database you are using.___
<br/><br/>

In [5]:
index_params = client.prepare_index_params()
index_params.add_index(field_name="vector", index_type="FLAT", metric_type="COSINE")

client.create_collection(
    collection_name='columns_metadata',
    schema=schema,
    index_params=index_params
)

<br/><br/>
___Now we will vectorize the metadata description in a for loop and store the vector embeddings in the vector database.___

<br/>

___I'll be using the local ollama model called ___nomic-embed-text___ for vectorization, you can use any other model for vectorization.___

<br/><br/>

In [6]:
milvus_data = []
for col in columns_metadata:

    # I'm using ollama -  A local model, you can use any other model to get the embeddings
    response = requests.post(
        "http://localhost:11434/api/embed",
        json={"model": "nomic-embed-text", "input": col["description"]}
    )

    vector_data = response.json()["embeddings"][0]

    milvus_data.append({
        "id": col["id"],
        "column_name": col["column_name"],
        "description": col["description"],
        "vector": vector_data  # The vector Embedding for meta data description
    })


<br/><br/>

In [9]:
milvus_data[2]

{'id': 3,
 'column_name': 'designation',
 'description': 'designation: job title, role, position, work title, occupation, department role',
 'vector': [-0.020311292,
  -0.012423892,
  -0.16541412,
  -0.038674787,
  0.0041727507,
  -0.027161332,
  -0.0017782061,
  -0.04826393,
  0.024908911,
  -0.02355217,
  -0.032993343,
  -0.005869006,
  0.06187705,
  -0.003061788,
  0.029358162,
  -0.05151753,
  -0.018941302,
  -0.061039083,
  -0.033833697,
  -0.0016383788,
  -0.020414906,
  -0.009388672,
  -0.016245741,
  0.007974595,
  0.067048885,
  0.06000586,
  0.0076138377,
  -0.050278395,
  0.031027576,
  -0.04875782,
  -0.020771565,
  0.065917864,
  0.0060459957,
  0.01755503,
  -0.024255961,
  -0.0822112,
  0.06716954,
  0.045545183,
  -0.041048165,
  0.024768015,
  -0.041111793,
  -0.04043552,
  0.002207169,
  -0.03805426,
  0.014260038,
  -0.014810569,
  0.019082928,
  0.04939748,
  0.054453835,
  -0.0031078635,
  -0.03060744,
  -0.04332079,
  0.011631403,
  0.0072989957,
  0.064177625,
  

<br/><br/>
___Insert the vectorized metadata into the vector database.___
<br/><br/>

In [12]:

client.insert(collection_name='columns_metadata', data=milvus_data)

{'insert_count': 3, 'ids': [1, 2, 3], 'cost': 0}

<br/><br/>
##### The Actual emoplyee database table in pandas dataframe

<br/>

___I've selcted pandas dataframs to store the data. This is just for demontration purpose, this could very well be a SQL database table___
<br/><br/>

In [13]:
# create a pandas dataframe for storing the employee table data of name, description and salary
df = pd.DataFrame([
  {
    "name": "Alice Smith",
    "designation": "Senior Cloud Solutions Architect",
    "salary": 145000.0
  },
  {
    "name": "Bob Jones",
    "designation": "Junior Frontend Developer",
    "salary": 65000.0
  },
  {
    "name": "Carol White",
    "designation": "Engineering Manager",
    "salary": 160000.0
  },
  {
    "name": "David Lee",
    "designation": "DevOps & Infrastructure Engineer",
    "salary": 125000.0
  },
  {
    "name": "Emma Davis",
    "designation": "Lead Machine Learning Engineer",
    "salary": 155000.0
  },
  {
    "name": "Frank Miller",
    "designation": "Database Administrator",
    "salary": 110000.0
  },
  {
    "name": "Grace Taylor",
    "designation": "Human Resources Director",
    "salary": 95000.0
  },
  {
    "name": "Henry Wilson",
    "designation": "Chief Technology Officer (CTO)",
    "salary": 220000.0
  },
  {
    "name": "Isabella Clark",
    "designation": "Senior QA & Automation Engineer",
    "salary": 102000.0
  },
  {
    "name": "Jack Robinson",
    "designation": "Product Manager",
    "salary": 130000.0
  }
]
)


<br/><br/>
___Here is how the pandas dataframe looks like.___

<br/><br/>

In [14]:
df

,name,designation,salary
0,Alice Smith,Senior Cloud Solutions Architect,145000.0
1,Bob Jones,Junior Frontend Developer,65000.0
2,Carol White,Engineering Manager,160000.0
3,David Lee,DevOps & Infrastructure Engineer,125000.0
4,Emma Davis,Lead Machine Learning Engineer,155000.0
5,Frank Miller,Database Administrator,110000.0
6,Grace Taylor,Human Resources Director,95000.0
7,Henry Wilson,Chief Technology Officer (CTO),220000.0
8,Isabella Clark,Senior QA & Automation Engineer,102000.0
9,Jack Robinson,Product Manager,130000.0


<br/><br/>
#### Using the LLM with RAG
<br/>

___Lets look at the question and use RAG to extract the relevant column metadata from the vector database and then use the LLM to answer the question based on the relevant column data.___


<br/><br/>

In [15]:
# let's ask the question, vectorize the question and find the most relevant column metadata from the vector database
question = "What is the highest compensation?"

<br/><br/>

In [16]:
# query the qiestion to the local ollama model to get the vector embedding for the question
#-----
# Make a note that the actual LLM interaction has not come int picture as yet. It will be happening after we have found the relevant column metadata from the vector database and then we will use the LLM to answer the question based on the relevant column data.


query_vector = requests.post(
    "http://localhost:11434/api/embed",
    json={"model": "nomic-embed-text", "input": question}
).json()["embeddings"][0]

<br/><br/>
___Serach the vector empbedding of the question in the vector database to find the most relevant column metadata.___
<br/><br/>

In [19]:
#search the query vector in the vector database for similarity search
search_res = client.search(
        collection_name='columns_metadata',
        data=[query_vector],
        limit=1,
        output_fields=["column_name"]
    )


<br/><br/>
___Get the resultant column___
<br/><br/>

In [20]:
search_res

data: [[{'id': 1, 'distance': 0.3935886025428772, 'entity': {'id': 1, 'column_name': 'salary'}}]]

In [21]:
matched_column = search_res[0][0]["entity"]["column_name"]

In [22]:
matched_column

'salary'

<br/><br/>
___Fetch the data from the matched column in the pandas dataframe and append the same in the question.___

<br/><br/>

In [23]:
# now ask the question to the LLM using the column data
relevant_data = df[matched_column]

In [24]:
relevant_data

0    145000.0
1     65000.0
2    160000.0
3    125000.0
4    155000.0
5    110000.0
6     95000.0
7    220000.0
8    102000.0
9    130000.0
Name: salary, dtype: float64

In [26]:
llm_prompt = f" Based on the Context: {relevant_data.to_list()} Question: {question}"

In [27]:
llm_prompt

' Based on the Context: [145000.0, 65000.0, 160000.0, 125000.0, 155000.0, 110000.0, 95000.0, 220000.0, 102000.0, 130000.0] Question: What is the highest compensation?'

<br/><br/>

___The LLM will now answer the question based on the relevant column data.___

<br/><br/>

In [28]:
llm_response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "qwen3.5:latest", "prompt": llm_prompt, "stream": False}
    )

<br/><br/>

In [29]:
llm_response.json()["response"]

'Based on the provided context list `[145000.0, 65000.0, 160000.0, 125000.0, 155000.0, 110000.0, 95000.0, 220000.0, 102000.0, 130000.0]`, the highest compensation is **$220,000**.'